# NSW Electricity Demand Forecasting with GenAI-Assisted Analysis

This portfolio notebook compares a persistence baseline, XGBoost and a tuned LSTM on a shared chronological test period. It then adds a small **GenAI answering layer** through Gradio.

The numerical work is always performed by Pandas. GenAI receives only a compact set of verified metrics and events, and turns that evidence into a clear explanation. This is a historical backtest, not a live operational forecast.

## How to run

1. Run the setup cell.
2. Leave `USE_GOOGLE_DRIVE = True` to use the full prediction file in Drive, or set it to `False` to use the public GitHub sample.
3. Add `OPENAI_API_KEY` to Colab Secrets if you want free-form GenAI answers. Never paste a key into a notebook that will be committed to GitHub.
4. Run the final cell to open the Gradio interface.

Without an API key, the notebook still produces the analysis and answers several common questions using deterministic calculations.

In [ ]:
%pip install -q "gradio>=6,<7" "openai>=3,<4" pyarrow

In [ ]:
from pathlib import Path
import sys

USE_GOOGLE_DRIVE = True
DRIVE_PARQUET_PATH = Path(
    "/content/drive/MyDrive/common_test_predictions.parquet"
)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

print(f"Running in Colab: {IN_COLAB}")
print(f"Configured Drive file: {DRIVE_PARQUET_PATH}")

## 1. Load and validate the prediction data

The loader understands both the full Google Drive schema and the smaller public GitHub schema.

In [ ]:
import pandas as pd

GITHUB_DEMO_URL = (
    "https://raw.githubusercontent.com/jagminderunsw/"
    "Nsw-Electricity-Demand-Forecasting/main/"
    "results/demo_predictions.parquet"
)

COLUMN_RENAME = {
    "DATETIME": "datetime",
    "Actual_Demand": "actual_mw",
    "Baseline_Prediction": "baseline_mw",
    "XGBoost_Prediction": "xgboost_mw",
    "LSTM_Prediction": "lstm_mw",
    "SelectedBaselineName": "selected_baseline",
}
REQUIRED_COLUMNS = [
    "datetime",
    "actual_mw",
    "baseline_mw",
    "xgboost_mw",
    "lstm_mw",
]


def normalise_predictions(frame):
    frame = frame.rename(columns=COLUMN_RENAME).copy()
    missing = sorted(set(REQUIRED_COLUMNS).difference(frame.columns))
    if missing:
        raise ValueError(f"Prediction data is missing columns: {missing}")

    frame["datetime"] = pd.to_datetime(frame["datetime"], errors="raise")
    frame = frame.sort_values("datetime").reset_index(drop=True)

    if frame["datetime"].duplicated().any():
        raise ValueError("Prediction data contains duplicate timestamps.")
    if frame[REQUIRED_COLUMNS].isna().any().any():
        raise ValueError("Prediction data contains missing evaluation values.")

    if "selected_baseline" not in frame:
        frame["selected_baseline"] = "Persistence Baseline"
    return frame


def find_drive_predictions():
    drive_root = Path("/content/drive/MyDrive")
    if not (USE_GOOGLE_DRIVE and drive_root.is_dir()):
        return []

    matches = sorted(drive_root.rglob("common_test_predictions.parquet"))
    if matches:
        print(f"Auto-detected Drive file: {matches[0]}")
    return matches


def load_predictions():
    local_candidates = [DRIVE_PARQUET_PATH] if USE_GOOGLE_DRIVE else []
    local_candidates.extend(find_drive_predictions())
    local_candidates.extend(
        [
            Path("results/demo_predictions.parquet"),
            Path("/content/demo_predictions.parquet"),
        ]
    )

    checked_paths = []
    for candidate in dict.fromkeys(local_candidates):
        checked_paths.append(str(candidate))
        if candidate.is_file():
            return normalise_predictions(pd.read_parquet(candidate)), str(candidate)

    try:
        demo = pd.read_parquet(GITHUB_DEMO_URL)
        return normalise_predictions(demo), GITHUB_DEMO_URL
    except Exception as error:
        searched = "\n- ".join(checked_paths)
        raise FileNotFoundError(
            "Could not locate common_test_predictions.parquet.\n\n"
            f"Paths checked:\n- {searched}\n\n"
            "Move the file anywhere under MyDrive and rerun this cell, or set "
            "DRIVE_PARQUET_PATH to its exact location in the settings cell. "
            "The public GitHub fallback becomes available after the updated "
            "repository is pushed."
        ) from error


predictions, DATA_SOURCE = load_predictions()

print(f"Loaded {len(predictions):,} observations from {DATA_SOURCE}")
print(
    f"Period: {predictions['datetime'].min():%d %b %Y %H:%M} to "
    f"{predictions['datetime'].max():%d %b %Y %H:%M}"
)
predictions.head()

## 2. Compare the forecasting models

Every model is evaluated on the same timestamps. Lower MAE, RMSE and MAPE are better; higher R-squared is better.

In [ ]:
import numpy as np

MODEL_COLUMNS = {
    "Persistence baseline": "baseline_mw",
    "XGBoost": "xgboost_mw",
    "LSTM": "lstm_mw",
}


def calculate_metrics(frame):
    actual = frame["actual_mw"].to_numpy(dtype="float64")
    rows = []

    for model, column in MODEL_COLUMNS.items():
        predicted = frame[column].to_numpy(dtype="float64")
        error = predicted - actual
        absolute_error = np.abs(error)
        total_variation = np.sum((actual - actual.mean()) ** 2)
        rows.append(
            {
                "Model": model,
                "Observations": len(frame),
                "MAE_MW": absolute_error.mean(),
                "RMSE_MW": np.sqrt(np.mean(error**2)),
                "MAPE_pct": np.mean(absolute_error / actual) * 100,
                "R2": 1 - np.sum(error**2) / total_variation,
            }
        )

    return pd.DataFrame(rows).sort_values("RMSE_MW").reset_index(drop=True)


metrics = calculate_metrics(predictions)
metrics.round({"MAE_MW": 2, "RMSE_MW": 2, "MAPE_pct": 2, "R2": 4})

In [ ]:
import plotly.graph_objects as go

chart_data = predictions.tail(14 * 48)
figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=chart_data["datetime"],
        y=chart_data["actual_mw"],
        name="Actual demand",
        line={"color": "#222222", "width": 2},
    )
)
for model, column, colour in [
    ("XGBoost", "xgboost_mw", "#E69F00"),
    ("LSTM", "lstm_mw", "#0072B2"),
]:
    figure.add_trace(
        go.Scatter(
            x=chart_data["datetime"],
            y=chart_data[column],
            name=model,
            line={"color": colour, "width": 1.4},
        )
    )

figure.update_layout(
    title="Actual versus predicted demand: final 14 days",
    xaxis_title="Datetime",
    yaxis_title="Demand (MW)",
    hovermode="x unified",
    template="plotly_white",
    height=520,
)
figure.show()

## 3. Build verified evidence for the answering layer

This cell converts the prediction table into a compact evidence package. The package contains model metrics, coverage, peak demand and the largest errors. The raw time-series rows are not sent to GenAI.

In [ ]:
import json


def build_verified_evidence(frame, metric_table):
    peak_row = frame.loc[frame["actual_mw"].idxmax()]
    worst_events = []

    for model, column in MODEL_COLUMNS.items():
        absolute_error = (frame[column] - frame["actual_mw"]).abs()
        row = frame.loc[absolute_error.idxmax()]
        worst_events.append(
            {
                "model": model,
                "datetime": row["datetime"].isoformat(),
                "actual_mw": round(float(row["actual_mw"]), 2),
                "predicted_mw": round(float(row[column]), 2),
                "absolute_error_mw": round(float(absolute_error.loc[row.name]), 2),
            }
        )

    xgb_rmse = float(
        metric_table.loc[metric_table["Model"] == "XGBoost", "RMSE_MW"].iloc[0]
    )
    lstm_rmse = float(
        metric_table.loc[metric_table["Model"] == "LSTM", "RMSE_MW"].iloc[0]
    )

    return {
        "scope": "Historical 30-minute-ahead NSW electricity-demand backtest",
        "data_source": (
            "full prediction file"
            if len(frame) > 90 * 48
            else "public 90-day demo sample"
        ),
        "coverage": {
            "observations": int(len(frame)),
            "start": frame["datetime"].min().isoformat(),
            "end": frame["datetime"].max().isoformat(),
        },
        "metrics": metric_table.round(4).to_dict(orient="records"),
        "lstm_rmse_reduction_vs_xgboost_pct": round(
            (xgb_rmse - lstm_rmse) / xgb_rmse * 100, 2
        ),
        "peak_actual_demand": {
            "datetime": peak_row["datetime"].isoformat(),
            "actual_mw": round(float(peak_row["actual_mw"]), 2),
        },
        "largest_absolute_error_by_model": worst_events,
        "limitations": [
            "Saved historical test predictions are used; this is not a live forecast.",
            "The evidence does not establish causal relationships.",
            "Questions outside the supplied evidence should be marked as unsupported.",
        ],
    }


verified_evidence = build_verified_evidence(predictions, metrics)
print(json.dumps(verified_evidence, indent=2)[:4000])

## 4. Configure the optional GenAI layer

In Google Colab, open the key icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and grant the notebook access. The key is read securely and is never printed.

In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if IN_COLAB:
    try:
        from google.colab import userdata

        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or OPENAI_API_KEY
    except Exception:
        pass

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
print("GenAI mode ready." if OPENAI_API_KEY else "No API key: verified fallback mode.")
print(f"Configured model: {OPENAI_MODEL}")

In [ ]:
from openai import OpenAI

SYSTEM_INSTRUCTIONS = """
You are the answering layer for a NSW electricity-demand forecasting portfolio project.
Answer only from VERIFIED_EVIDENCE supplied with the question. Never invent a metric,
timestamp, weather condition, causal explanation or live forecast. State which model
and metric support a comparison. If the evidence is insufficient, say so directly and
suggest an analysis that would be needed. Keep answers concise and understandable to
a hiring manager. Always describe the results as a historical backtest.
""".strip()


def deterministic_answer(question):
    question_lower = question.lower()
    best = metrics.iloc[0]

    if "lstm" in question_lower and "xgboost" in question_lower:
        lstm = metrics.loc[metrics["Model"] == "LSTM"].iloc[0]
        xgboost = metrics.loc[metrics["Model"] == "XGBoost"].iloc[0]
        reduction = (xgboost["RMSE_MW"] - lstm["RMSE_MW"]) / xgboost["RMSE_MW"] * 100
        return (
            f"On this historical backtest, LSTM RMSE was {lstm['RMSE_MW']:.2f} MW "
            f"versus {xgboost['RMSE_MW']:.2f} MW for XGBoost. "
            f"That is a {reduction:.2f}% reduction in RMSE."
        )

    if any(word in question_lower for word in ["best", "compare", "perform"]):
        return (
            f"On this historical backtest, {best['Model']} performed best by RMSE "
            f"({best['RMSE_MW']:.2f} MW). Its MAE was {best['MAE_MW']:.2f} MW, "
            f"MAPE was {best['MAPE_pct']:.2f}%, and R-squared was {best['R2']:.4f}."
        )

    if any(phrase in question_lower for phrase in ["largest error", "worst error"]):
        event = max(
            verified_evidence["largest_absolute_error_by_model"],
            key=lambda item: item["absolute_error_mw"],
        )
        return (
            f"The largest recorded model error in the loaded historical sample was "
            f"{event['absolute_error_mw']:.2f} MW for {event['model']} at "
            f"{event['datetime']}. Actual demand was {event['actual_mw']:.2f} MW."
        )

    if any(word in question_lower for word in ["period", "coverage", "observation"]):
        coverage = verified_evidence["coverage"]
        return (
            f"The loaded historical backtest contains {coverage['observations']:,} "
            f"observations from {coverage['start']} to {coverage['end']}."
        )

    if "limit" in question_lower or "live" in question_lower:
        return " ".join(verified_evidence["limitations"])

    return (
        "I can calculate model comparisons, coverage, largest errors and limitations "
        "without an API key. Add OPENAI_API_KEY to Colab Secrets for free-form "
        "questions grounded in the same verified evidence."
    )


def recent_history_text(history, maximum_messages=6):
    lines = []
    for item in history[-maximum_messages:]:
        role = str(item.get("role", "unknown"))
        content = item.get("content", "")
        if isinstance(content, str):
            lines.append(f"{role}: {content[:500]}")
    return "\n".join(lines)


def ask_forecast(message, history):
    question = str(message).strip()
    if not question:
        return "Please ask a question about the forecasting results."
    if len(question) > 500:
        return "Please keep the question below 500 characters."
    if not OPENAI_API_KEY:
        return deterministic_answer(question)

    client = OpenAI(api_key=OPENAI_API_KEY)
    prompt = f"""
VERIFIED_EVIDENCE
{json.dumps(verified_evidence, indent=2)}

RECENT_CONVERSATION
{recent_history_text(history)}

QUESTION
{question}
""".strip()

    try:
        response = client.responses.create(
            model=OPENAI_MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=prompt,
            max_output_tokens=400,
        )
        return response.output_text
    except Exception as error:
        fallback = deterministic_answer(question)
        return (
            f"The GenAI request could not be completed ({type(error).__name__}). "
            f"Verified fallback answer:\n\n{fallback}"
        )

## 5. Launch the Gradio answering interface

Try one of the example questions, then ask your own. The interface answers only about the loaded historical forecasting evidence.

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=ask_forecast,
    title="NSW Electricity Forecasting Q&A",
    description=(
        "Ask about model performance, test coverage, forecast errors and "
        "project limitations. Answers are grounded in verified Pandas calculations."
    ),
    examples=[
        "Which model performed best?",
        "Compare LSTM and XGBoost using RMSE.",
        "What was the largest forecasting error?",
        "What period does this analysis cover?",
        "What are the limitations of this project?",
    ],
    save_history=False,
)

demo.launch(share=IN_COLAB)

## Responsible-use notes

- This assistant explains saved backtest results; it does not make new demand forecasts.
- Pandas calculates every numerical fact before the GenAI call.
- The raw prediction dataset is not included in the prompt.
- Unsupported questions should receive an explicit limitation rather than a fabricated answer.
- API keys belong in Colab Secrets or environment variables, never in GitHub.